In [1]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split
from collections import Counter

# --- 1. Configuration (Based on Paper & Notebook) ---
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Hyperparameters based on the paper's experimental setup
IMG_SIZE = 224
BATCH_SIZE = 8
EPOCHS = 30
LR = 0.00001
NUM_CLASSES = 10 # 10 classes for COD10K/ERVA 1.0 (Pipefish, Seahorse, etc.)

# --- 2. Dataset Paths (References from the uploaded notebook) ---
# NOTE: These paths must be valid in your execution environment.
train_dir_cod = "/kaggle/input/cod10k/COD10K-v3/Train" 
test_dir_cod = "/kaggle/input/cod10k/COD10K-v3/Test"
train_dir_camo_cam = "/kaggle/input/camo-coco/CAMO_COCO/Camouflage"
train_dir_camo_noncam = "/kaggle/input/camo-coco/CAMO_COCO/Non_Camouflage"
testing_images_dir = "/kaggle/input/testing-dataset/Images"

# Fictional TXT files for demonstration (replace with actual path logic if needed)
# Since the paper implies using specific species, a custom label mapping would be needed,
# but for a complete pipeline, we keep the MultiDataset structure from the notebook.
ALL_ROOT_DIRS = [train_dir_cod, test_dir_cod, train_dir_camo_cam, train_dir_camo_noncam, testing_images_dir]

# --- 3. Data Utility Functions (Adapted from the notebook to support 10 classes) ---

def read_file_with_encoding(file_path, encodings=['utf-8', 'utf-8-sig', 'ISO-8859-1']):
    """Reads a file using a list of common encodings."""
    for encoding in encodings:
        try:
            with open(file_path, 'r', encoding=encoding) as f:
                return f.readlines()
        except UnicodeDecodeError:
            continue
    raise RuntimeError(f"Unable to read {file_path} with any of the provided encodings.")

# Simple transformation pipeline for consistency with the paper's data augmentation mention
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(15), # Random rotation used for data augmentation
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# NOTE: The notebook used a complex MultiDataset class with .txt files and different data paths.
# To make this code runnable, we define a simple placeholder. In a real scenario, you would
# need to adapt your full data loading logic (MultiDataset class, label mapping to 10 classes).

class SimpleCamouflageDataset(Dataset):
    """
    Placeholder class. In a real scenario, this replaces the MultiDataset in the notebook
    and should handle mapping file paths to 10 class indices (0-9).
    """
    def __init__(self, data_list, transform=None):
        # data_list should be a list of tuples: (image_path, class_index_0_to_9)
        self.data_list = data_list
        self.transform = transform

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        img_path, label = self.data_list[idx]
        img = Image.open(img_path).convert("RGB")
        
        if self.transform:
            img = self.transform(img)
            
        return img, label

# --- 4. The Proposed DenseNet201 + MobileNet Model (Fig. 1 Implementation) ---

class DenseNetMobileNetFusion(nn.Module):
    """
    Implements the dual-branch feature concatenation architecture 
    based strictly on Fig. 1 of the paper.
    """
    def __init__(self, num_classes=10):
        super().__init__()
        
        # --- DenseNet201 Branch ---
        # Load pre-trained DenseNet201, keeping features and freezing weights
        densenet = models.densenet201(pretrained=True)
        self.densenet_features = densenet.features
        for param in self.densenet_features.parameters():
            param.requires_grad = False
            
        # Get the number of features after DenseNet201's features block
        # This is the output channel count before GlobalAveragePooling2D
        # For DenseNet201, the feature output is 1920 channels
        dense_out_channels = densenet.classifier.in_features # 1920

        # Define the MLP head for the DenseNet201 branch
        # [cite_start]GlobalAveragePooling2D -> Dense 512 -> Dense 128 -> Dense 10 [cite: 1590, 1592, 1597, 1603]
        self.densenet_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), # GlobalAveragePooling2D
            nn.Flatten(),
            nn.Linear(dense_out_channels, 512), # Dense 512
            nn.ReLU(),
            nn.Dropout(0.2), # Dropout 0.2
            nn.Linear(512, 128), # Dense 128
            nn.ReLU(),
            nn.Dropout(0.5), # Dropout 0.5
            nn.Linear(128, num_classes), # Dense 10 (intermediate output)
        )
        
        # --- MobileNetV2 Branch (using v2 for standard feature count) ---
        # Load pre-trained MobileNetV2, keeping features and freezing weights
        # The paper used MobileNet (presumably V1 or V2/V3), V2 is a good standard proxy.
        mobilenet = models.mobilenet_v2(pretrained=True)
        self.mobilenet_features = mobilenet.features
        for param in self.mobilenet_features.parameters():
            param.requires_grad = False
        
        # MobileNetV2 features out at 1280 channels
        mobile_out_channels = mobilenet.last_channel # 1280

        # Define the MLP head for the MobileNet branch
        # [cite_start]GlobalAveragePooling2D -> Dense 512 -> Dense 128 -> Dense 10 [cite: 1591, 1594, 1601, 1613]
        self.mobilenet_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), # GlobalAveragePooling2D
            nn.Flatten(),
            nn.Linear(mobile_out_channels, 512), # Dense 512
            nn.ReLU(),
            nn.Dropout(0.2), # Dropout 0.2
            nn.Linear(512, 128), # Dense 128
            nn.ReLU(),
            nn.Dropout(0.5), # Dropout 0.5
            nn.Linear(128, num_classes), # Dense 10 (intermediate output)
        )
        
        # --- Concatenation and Final Classification Head ---
        # Concatenate outputs of the two Dense 10 layers (10 + 10 = 20 features)
        # [cite_start]-> Dense 512 -> Dense 128 -> Dense 10 -> Softmax [cite: 1604, 1605, 1608, 1611, 1614]
        self.classifier_head = nn.Sequential(
            nn.Linear(num_classes * 2, 512), # Dense 512 (Input is 20, 10 from each branch)
            nn.ReLU(),
            nn.Dropout(0.2), # Dropout 0.2
            nn.Linear(512, 128), # Dense 128
            nn.ReLU(),
            nn.Dropout(0.5), # Dropout 0.5
            nn.Linear(128, num_classes), # Final Dense 10
            # Softmax is often part of the loss function (CrossEntropyLoss) in PyTorch,
            # but is explicitly shown in the diagram, so we include it for completeness.
            nn.Softmax(dim=1) 
        )

    def forward(self, x):
        # DenseNet Branch
        d_feats = self.densenet_features(x)
        d_out = self.densenet_head(d_feats)
        
        # MobileNet Branch
        m_feats = self.mobilenet_features(x)
        m_out = self.mobilenet_head(m_feats)
        
        # [cite_start]Concatenate features [cite: 1604]
        combined_features = torch.cat((d_out, m_out), dim=1)
        
        # Final Classification
        output = self.classifier_head(combined_features)
        
        return output

# --- 5. Training Pipeline Structure ---

def train_and_validate_model(model, train_loader, val_loader, criterion, optimizer, epochs, device):
    best_val_accuracy = 0.0
    
    for epoch in range(epochs):
        # Training Phase
        model.train()
        running_loss = 0.0
        
        for inputs, labels in train_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            
        train_loss = running_loss / len(train_loader.dataset)

        # Validation Phase
        model.eval()
        corrects = 0
        total = 0
        val_loss = 0.0
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(device)
                labels = labels.to(device)
                
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                
                _, preds = torch.max(outputs, 1)
                corrects += torch.sum(preds == labels.data)
                total += labels.size(0)

        val_accuracy = corrects.double() / total
        val_loss = val_loss / len(val_loader.dataset)
        
        print(f'Epoch {epoch+1}/{epochs}')
        print(f'Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_accuracy:.4f}')

        # Save the best model
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            # torch.save(model.state_dict(), 'best_fusion_model.pth')
            # print("Saved best model.")

# --- Initialization and Execution ---

# 1. Initialize the Model
model = DenseNetMobileNetFusion(num_classes=NUM_CLASSES).to(device)

# [cite_start]2. Define Loss and Optimizer (using Categorical Cross-Entropy [cite: 1951])
# NOTE: PyTorch's CrossEntropyLoss includes Softmax, so the Softmax layer
# in the model is technically redundant for this loss function but kept in the
# model class for fidelity to the diagram.
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# 3. Create Dummy DataLoaders (Replace with your actual data loading)
# This uses dummy data because the raw paths from the notebook cannot be accessed/processed here.
dummy_data = [(os.path.join(testing_images_dir, f'dummy_{i}.jpg'), random.randint(0, 9)) for i in range(100)]
train_data, val_data = train_test_split(dummy_data, test_size=0.2, random_state=42)

train_ds = SimpleCamouflageDataset(train_data, transform=train_transform)
val_ds = SimpleCamouflageDataset(val_data, transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

print("\n--- Model Summary ---")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f} Million")


Device: cuda


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet201_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet201_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/densenet201-c1103571.pth" to /root/.cache/torch/hub/checkpoints/densenet201-c1103571.pth
100%|██████████| 77.4M/77.4M [00:00<00:00, 229MB/s]
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may 


--- Model Summary ---
Total parameters: 22.17 Million


# Dual branch 